# Forecasting with ChakraTab: drivers as columns

ChakraTS forecasts a series from its history. When most of what happens is explained by things you **know in advance** (paydays, promotions, holidays, the day of the week), you can also forecast as a **table**: one row per series and day, the known drivers as columns, and **ChakraTab** learning how each one moves the target.

This recipe:

1. builds that table from a daily history,
2. forecasts 28 days with ChakraTab, with intervals,
3. compares it with ChakraTS and with the average of the two,
4. scores all three at several past dates (a backtest), so you choose on evidence,
5. shows what each driver does to the forecast.

You need an API key in `YHAT_API_KEY` ([get one](https://yhatlabs.com/models)). The data is synthetic, so the notebook runs anywhere.

In [1]:
# pip install -U "yhatlabs[plot]"
import warnings
import numpy as np
import pandas as pd
import yhatlabs
from yhatlabs import Client

warnings.filterwarnings("ignore", category=DeprecationWarning)  # numpy 2 notices from pandas date arithmetic
yh = Client()
print("yhatlabs", yhatlabs.__version__)

yhatlabs 0.2.2


## The data

Daily orders at three stores for three years. Orders jump on paydays (the 1st and the 15th) and on promotion days, dip on Sundays, and each store has its own size. Promotions are planned ahead, so their future dates are known.

In [2]:
days = pd.date_range("2023-01-01", periods=3 * 365, freq="D")
rng = np.random.default_rng(7)
frames = []
for store, base in (("north", 200), ("south", 120), ("east", 80)):
    payday = days.day.isin([1, 15]).astype(float)
    promo = (rng.random(len(days)) < 0.06).astype(float)          # planned promotion days
    sunday = (days.dayofweek == 6).astype(float)
    orders = base * (1 + 0.5 * payday + 0.8 * promo - 0.3 * sunday) * (1 + rng.normal(0, 0.05, len(days)))
    frames.append(pd.DataFrame({"unique_id": store, "ds": days, "y": orders.round(1), "payday": payday, "promo": promo}))
data = pd.concat(frames, ignore_index=True)
data.head()

,unique_id,ds,y,payday,promo
0,north,2023-01-01,228.6,1.0,0.0
1,north,2023-01-02,198.9,0.0,0.0
2,north,2023-01-03,184.0,0.0,0.0
3,north,2023-01-04,214.7,0.0,0.0
4,north,2023-01-05,175.9,0.0,0.0


## Build the table

One row per store and day. The columns are what is known for every day, past or future, plus where the store stands:

| Column | Why |
|---|---|
| `unique_id` | which store: one table holds every store, so they learn from each other |
| `dow`, `dom`, `month` | the weekly and monthly pattern |
| `payday`, `promo`, `days_to_payday` | the known drivers |
| `level` | the mean of the 28 days before; **frozen at the forecast origin** for the days ahead, so no future value leaks in |
| `ratio` (target) | the day's value divided by its level, so stores of different sizes share one model |

In [3]:
FEATURES = ["unique_id", "dow", "dom", "month", "payday", "promo", "days_to_payday", "level"]


def table(data, origin):
    """Rows before `origin` have their target; rows from `origin` on are the days to forecast."""
    out = data.assign(y=data.y.where(data.ds < origin))
    out["dow"], out["dom"], out["month"] = out.ds.dt.dayofweek, out.ds.dt.day, out.ds.dt.month
    out["days_to_payday"] = np.where(out.ds.dt.day <= 15, 15 - out.ds.dt.day, out.ds.dt.days_in_month + 1 - out.ds.dt.day)
    out["level"] = out.groupby("unique_id").y.transform(lambda y: y.shift(1).rolling(28, min_periods=7).mean().ffill())
    out["ratio"] = out.y / out.level
    return out.dropna(subset=["level"]).reset_index(drop=True)


def chakratab_forecast(data, origin, h, model="chakra-tab"):
    t = table(data, origin)
    train = t[t.ds < origin]
    future = t[(t.ds >= origin) & (t.ds < origin + pd.Timedelta(days=h))].reset_index(drop=True)
    res = yh.predict(train[FEATURES + ["ratio"]], "ratio", future[FEATURES], quantiles=[0.1, 0.9], model=model)
    return future[["unique_id", "ds"]].assign(
        q10=res.df["q10"].to_numpy() * future.level.to_numpy(),
        median=res.df["prediction"].to_numpy() * future.level.to_numpy(),
        q90=res.df["q90"].to_numpy() * future.level.to_numpy(),
    )


h = 28
origin = days[-h]                                    # hold the last 28 days back to score on
fc_tab = chakratab_forecast(data, origin, h)
fc_tab.head()

,unique_id,ds,q10,median,q90
0,north,2025-12-03,186.538911,199.634245,212.793363
1,north,2025-12-04,186.091533,199.279692,212.392337
2,north,2025-12-05,186.235552,199.316071,212.620085
3,north,2025-12-06,186.678731,199.880507,213.191799
4,north,2025-12-07,131.349894,140.431749,149.326789


## Compare with ChakraTS, and with the average of the two

ChakraTS gets the same drivers as known covariates. Both forecasts are scored on the 28 days held back with WAPE (the error as a share of the actuals; lower is better).

In [4]:
def chakrats_forecast(data, origin, h):
    history = data[data.ds < origin]
    future = data[(data.ds >= origin) & (data.ds < origin + pd.Timedelta(days=h))].drop(columns="y")
    return yh.forecast(history, h, known=["payday", "promo"], future=future).df


def wape(fc, data):
    m = fc.merge(data[["unique_id", "ds", "y"]], on=["unique_id", "ds"])
    return np.abs(m["median"] - m.y).sum() / m.y.abs().sum()


def average(tab, ts):
    m = tab.merge(ts[["unique_id", "ds", "median"]], on=["unique_id", "ds"], suffixes=("", "_ts"))
    return m.assign(median=(m["median"] + m["median_ts"]) / 2).drop(columns="median_ts")


fc_ts = chakrats_forecast(data, origin, h)
fc_avg = average(fc_tab, fc_ts)
pd.Series({"ChakraTab": wape(fc_tab, data), "ChakraTS": wape(fc_ts, data), "average": wape(fc_avg, data)}, name="WAPE, last 28 days").round(3)

ChakraTab    0.042
ChakraTS     0.045
average      0.041
Name: WAPE, last 28 days, dtype: float64

## Score at several past dates before choosing

One 28-day window can flatter either approach. A backtest forecasts from several past origins, each time using only the data before it.

In [5]:
rows = []
for origin_i in pd.date_range(days[-h] - pd.DateOffset(months=5), days[-h], freq="MS"):
    tab, ts = chakratab_forecast(data, origin_i, h), chakrats_forecast(data, origin_i, h)
    avg = average(tab, ts)
    rows.append({"origin": origin_i.date(), "ChakraTab": wape(tab, data), "ChakraTS": wape(ts, data), "average": wape(avg, data)})
backtest = pd.DataFrame(rows).set_index("origin")
backtest.loc["mean"] = backtest.mean()
backtest.round(3)

,ChakraTab,ChakraTS,average
origin,,,
2025-08-01,0.043,0.054,0.046
2025-09-01,0.043,0.056,0.047
2025-10-01,0.042,0.050,0.044
2025-11-01,0.036,0.050,0.041
2025-12-01,0.038,0.046,0.039
mean,0.040,0.051,0.043


## What each driver does

Fitting once keeps the model, so you can ask how the forecast moves with each column (`effect`) and which columns matter most (`importance`).

In [6]:
t = table(data, origin)
train = t[t.ds < origin]
with yh.fit(train[FEATURES + ["ratio"]], "ratio") as model:
    print(model.effect("promo", grid=[0, 1])[["promo", "average"]].round(2).to_string(index=False))
    print(model.effect("dow", grid=list(range(7)))[["dow", "average"]].round(2).to_string(index=False))

 promo  average
     0     0.95
     1     1.72


 dow  average
   0     1.03
   1     1.03
   2     1.02
   3     1.02
   4     1.03
   5     1.03
   6     0.74


Promotion days lift orders by roughly the 80% built into the data, and Sundays (`dow` 6) sit about 30% below the other days: ChakraTab recovered both from the table alone.

## ChakraTab v2

If your organization has ChakraTab v2 (in private preview, strongest on wide tables), it takes the same calls. `yh.models()` lists the models your key can use.

In [7]:
if "chakra-tab-v2" in [m["id"] for m in yh.models()]:
    fc_v2 = chakratab_forecast(data, origin, h, model="chakra-tab-v2")
    print("ChakraTab v2 WAPE:", round(wape(fc_v2, data), 3))
else:
    print("chakra-tab-v2 is not enabled for this key")

ChakraTab v2 WAPE: 0.042


## When to use which

- **ChakraTS** (`yh.forecast`) is the default: trends, changing seasonality, long histories, calibrated intervals at every step. [Tuning](https://yhatlabs.com/models/docs/time-series/tuning) adapts it to your own series and drivers.
- **ChakraTab on a table** shines when known drivers dominate and the level is stable, and it tells you each driver's effect. It does not extrapolate a trend beyond the frozen level.
- **The average of the two** is often better than either: check it in your backtest.

More: [Forecasting with ChakraTab](https://yhatlabs.com/models/docs/time-series/forecasting-with-chakratab) in the docs.